# Fine-tuning YOLO11n sur American Sign Language Letters

**Plateforme :** Google Colab (Runtime > Change runtime type > T4 GPU)

**Objectif :** Entraîner un modèle YOLO11n sur le dataset ASL (26 lettres A-Z) pour la détection en temps réel.

**Durée estimée :** ~1h30-2h sur T4.

## 1. Vérification GPU

In [1]:
!nvidia-smi

Fri May  8 16:38:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installation des dépendances

In [2]:
!pip install -q ultralytics==8.3.* roboflow supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 132.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00


## 3. Téléchargement du dataset ASL via Roboflow

Dataset public : **American Sign Language Letters** par David Lee (26 classes, ~1700 images, déjà splité train/val/test).

In [3]:
import os
from roboflow import Roboflow

API_KEY = os.environ.get("ROBOFLOW_API_KEY", "01ppKST0VVzBcfnAElsw")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
version = project.version(6)
dataset = version.download("yolov11")

print("Dataset téléchargé dans:", dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to American-Sign-Language-Letters-6 in yolov11:: 100%|██████████| 1452/1452 [00:00<00:00, 3101.74it/s]

Dataset téléchargé dans: /content/American-Sign-Language-Letters-6


In [4]:
# Vérifier le data.yaml
import yaml
from pathlib import Path

data_yaml = Path(dataset.location) / "data.yaml"
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
print(cfg)
print(f"\nNombre de classes: {cfg['nc']}")
print(f"Classes: {cfg['names']}")

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 26, 'names': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z'], 'roboflow': {'workspace': 'david-lee-d0rhs', 'project': 'american-sign-language-letters', 'version': 6, 'license': 'Public Domain', 'url': 'https://universe.roboflow.com/david-lee-d0rhs/american-sign-language-letters/dataset/6'}}

Nombre de classes: 26
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


## 4. Entraînement YOLO11n

- 50 epochs, image size 640, batch 32
- Patience 15 (early stopping)
- Sauvegarde tous les 10 epochs
- Augmentations par défaut d'Ultralytics (mosaic, mixup, hsv, flips)

In [5]:
from ultralytics import YOLO
import torch

torch.manual_seed(42)

model = YOLO('yolo11n.pt')

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=32,
    patience=15,
    save_period=10,
    project='runs/detect',
    name='yolo11n_asl',
    seed=42,
    plots=True,
    verbose=True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.47 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/American-Sign-Language-Letters-6/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=t

## 5. Évaluation sur le test set

In [7]:
!find runs -name 'best.pt' 2>/dev/null

runs/detect/runs/detect/yolo11n_asl/weights/best.pt


In [8]:
import glob, os
from ultralytics import YOLO

# Cherche best.pt récursivement où qu'il soit
candidates = sorted(glob.glob('**/yolo11n_asl*/weights/best.pt', recursive=True),
                    key=os.path.getmtime, reverse=True)
assert candidates, "Aucun best.pt trouvé"
best_path = candidates[0]
print(f"Utilisation de: {best_path}")

best_model = YOLO(best_path)
metrics = best_model.val(data=str(data_yaml), split='test', plots=True)
print(f"\n=== Résultats sur test set ===")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

Utilisation de: runs/detect/runs/detect/yolo11n_asl/weights/best.pt
Ultralytics 8.3.253 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,587,222 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1783.2±907.7 MB/s, size: 225.3 KB)
val: Scanning /content/American-Sign-Language-Letters-6/test/labels... 72 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 2.0Kit/s 0.0s
val: New cache created: /content/American-Sign-Language-Letters-6/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
                   all         72         72      0.879      0.755      0.904      0.876
                     A          1          1      0.772          1      0.995      0.995
                     B          3          3      0.959          1      0.995      0.995
                     C          4          4      0.985   

## 6. Mesure du temps d'inférence

In [9]:
import time
import glob
from PIL import Image

test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")[:50]
print(f"Mesure sur {len(test_images)} images")

_ = best_model.predict(test_images[0], verbose=False)  # warmup

start = time.time()
for img in test_images:
    _ = best_model.predict(img, verbose=False)
elapsed = time.time() - start
print(f"Temps moyen par image: {1000*elapsed/len(test_images):.2f} ms")
print(f"FPS estimé: {len(test_images)/elapsed:.1f}")

Mesure sur 50 images
Temps moyen par image: 23.46 ms
FPS estimé: 42.6


## 7. Sauvegarde et téléchargement du modèle

In [11]:
import shutil, os
from google.colab import files

# best_path est défini dans la cellule d'évaluation précédente.
# Si tu n'as pas exécuté la cellule d'éval, décommente :
# import glob
# best_path = sorted(glob.glob('**/yolo11n_asl*/weights/best.pt', recursive=True),
#                    key=os.path.getmtime, reverse=True)[0]

size_mb = os.path.getsize(best_path) / 1024 / 1024
print(f"Taille modèle best.pt: {size_mb:.2f} MB")

run_dir = os.path.dirname(os.path.dirname(best_path))  # weights/best.pt -> run dir
print(f"Run dir: {run_dir}")

shutil.make_archive('yolo11n_asl_run', 'zip', run_dir)
print("Archive créée: yolo11n_asl_run.zip")

files.download(best_path)
files.download('yolo11n_asl_run.zip')


Taille modèle best.pt: 5.21 MB
Run dir: runs/detect/runs/detect/yolo11n_asl
Archive créée: yolo11n_asl_run.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>